# 01. Preprocesamiento y Limpieza de Datos

Pipeline autónomo para procesar archivos crudos desde `data/raw/` hacia archivos Parquet limpios en `data/interim/`.

El procesamiento detecta automáticamente las carpetas en `data/raw/` (`enosa`, `hidrandina`), maneja archivos `.zip` en memoria sin extraerlos a disco, y aplica las heurísticas de reparación registradas en `metadata/datasets.json`:
- `corrupted_csv`: repara comillas desbalanceadas, tabulaciones y líneas partidas. Guarda en bitácora las líneas crudas antes y la línea armada después.
- `corrupted_xlsx`: desagrega registros concatenados dentro de celdas con saltos de línea.
- `extra_line`: extrae y aísla a nivel de fila cualquier registro fuera de formato hacia la bitácora sin descartar el resto del archivo.
- `csv` / `xlsx`: lectura estándar con normalización de esquema.


In [10]:
from __future__ import annotations

import csv
import io
import json
import time
import zipfile
import re
from pathlib import Path

import polars as pl
import pandas as pd
from IPython.display import display

# Rutas
ROOT = Path.cwd().resolve()
RAW_BASE_DIR = ROOT / "data" / "raw"
INTERIM_BASE_DIR = ROOT / "data" / "interim"
METADATA_PATH = ROOT / "metadata" / "datasets.json"

# Detección automática de carpetas en data/raw
carpetas_raw = sorted([d.name for d in RAW_BASE_DIR.iterdir() if d.is_dir()])

print(f"Raíz del proyecto:    {ROOT}")
print(f"Carpeta RAW base:     {RAW_BASE_DIR}")
print(f"Carpetas detectadas:  {carpetas_raw}")
print(f"Carpeta INTERIM base: {INTERIM_BASE_DIR}")
print(f"Catálogo metadatos:   {METADATA_PATH}")


Raíz del proyecto:    /home/Yair/CodeProjects/MachineLearning/Project
Carpeta RAW base:     /home/Yair/CodeProjects/MachineLearning/Project/data/raw
Carpetas detectadas:  ['enosa', 'hidrandina']
Carpeta INTERIM base: /home/Yair/CodeProjects/MachineLearning/Project/data/interim
Catálogo metadatos:   /home/Yair/CodeProjects/MachineLearning/Project/metadata/datasets.json


## 1. Esquemas de Columnas

Definición de las 25 columnas canónicas, tipos de datos esperados y lista de columnas a descartar (PII y metadatos no analíticos). En Parquet se conservan 11 columnas.


In [11]:
COLUMNS = [
    "FECHA_CORTE", "NRO_SERVICIO", "NOMBRE_SERVICIO", "DIRECCION",
    "ID_TIPO_IDENTIDAD", "NRO_IDENTIDAD", "ID_EMPRESA", "NOMBRE_EMPRESA",
    "ID_UUNN", "UNIDAD_NEGOCIO", "PERIODO", "TARIFA", "DEPARTAMENTO",
    "PROVINCIA", "DISTRITO", "UBIGEO", "ID_TIPO_DOC_FAC", "NRO_DOC_FAC",
    "CARTERA", "FECHA_VENCIMIENTO", "FECHA_EMISION",
    "FECHA_CONSUMO_DESDE", "FECHA_CONSUMO_HASTA", "IMPORTE", "CONSUMO",
]

LEGACY_COLUMNS = [
    "fechacorte", "IdNroServicio", "NombreNroServicio", "Direccion",
    "IdTipoIdentidad", "NroIdentidad", "IdEmpresa", "NombreEmpresa",
    "IdUUNN", "nombreunidadnegocio", "Periodo", "Tarifa", "Departamento",
    "Provincia", "Distrito", "Ubigeo", "idTipoDocumento", "NroDocumento",
    "Cartera", "FechaVencimiento", "FechaEmision", "FechaConsumoDesde",
    "FechaConsumoHasta", "Importe", "ConsumosCalculados",
]

DISTRILUZ_EXCEL_COLUMNS = [
    "FECHA_CORTE", "NRO_SERVICIO", "NOMBRE_SERVICIO", "DIRECCION",
    "ID_TIPO_IDENTIDAD", "NRO_IDENTIDAD", "ID_EMPRESA", "NOMBRE_EMPRESA",
    "ID_UUNN", "UNIDAD_NEGOCIO", "PERIODO", "TARIFA", "DEPARTAMENTO",
    "PROVINCIA", "DISTRITO", "UBIGEO", "ID_TIPO_DOC_FAC", "NRO_DOC_FAC",
    "CARTERA", "FECHA_VENCIMIENTO", "FECHA_EMISION",
    "FECHA_COSNUMO_DESDE", "FECHA_CONSUMO_HASTA", "IMPORTE", "CONSUMO",
]

LEGACY_TO_COLUMNS = dict(zip(LEGACY_COLUMNS, COLUMNS))
NORMALIZED_MAP = {col.lower(): col for col in COLUMNS}
NORMALIZED_MAP.update({leg.lower(): col for leg, col in zip(LEGACY_COLUMNS, COLUMNS)})
NORMALIZED_MAP.update({
    "fecha_cosnumo_desde": "FECHA_CONSUMO_DESDE",
    "fecha_cosnumo_hasta": "FECHA_CONSUMO_HASTA",
})

SCHEMA_LEGACY_STRING = {column: pl.String for column in LEGACY_COLUMNS}
SCHEMA_STANDARD_STRING = {column: pl.String for column in DISTRILUZ_EXCEL_COLUMNS}
SCHEMA_PL_STRING = SCHEMA_LEGACY_STRING

SCHEMA_VAL = {
    "FECHA_CORTE": pl.Date, "NRO_SERVICIO": pl.String,
    "NOMBRE_SERVICIO": pl.String, "DIRECCION": pl.String,
    "ID_TIPO_IDENTIDAD": pl.Int8, "NRO_IDENTIDAD": pl.String,
    "ID_EMPRESA": pl.Int8, "NOMBRE_EMPRESA": pl.String,
    "ID_UUNN": pl.Int32, "UNIDAD_NEGOCIO": pl.String, "PERIODO": pl.Int32,
    "TARIFA": pl.String, "DEPARTAMENTO": pl.String, "PROVINCIA": pl.String,
    "DISTRITO": pl.String, "UBIGEO": pl.String, "ID_TIPO_DOC_FAC": pl.Int8,
    "NRO_DOC_FAC": pl.String, "CARTERA": pl.String,
    "FECHA_VENCIMIENTO": pl.Date, "FECHA_EMISION": pl.Date,
    "FECHA_CONSUMO_DESDE": pl.Date, "FECHA_CONSUMO_HASTA": pl.Date,
    "IMPORTE": pl.Float32, "CONSUMO": pl.Float32,
}

DROP_COLUMNS = [
    "FECHA_CORTE", "NRO_SERVICIO", "NOMBRE_SERVICIO", "DIRECCION",
    "ID_TIPO_IDENTIDAD", "NRO_IDENTIDAD", "ID_EMPRESA", "NOMBRE_EMPRESA",
    "ID_TIPO_DOC_FAC", "NRO_DOC_FAC",
    "FECHA_CONSUMO_HASTA", "FECHA_CONSUMO_DESDE", "FECHA_EMISION",
    "FECHA_VENCIMIENTO"
]

print(f"Columnas canónicas: {len(COLUMNS)}")
print(f"Columnas finales en Parquet: {len(COLUMNS) - len(DROP_COLUMNS)}")


Columnas canónicas: 25
Columnas finales en Parquet: 11


## 2. Ingesta de Archivos Crudos y Manejo de ZIP

Función para abrir archivos directamente desde `data/raw/{distribuidora}/`. Si el archivo viene en `.zip`, lo descomprime en memoria y entrega su flujo de bytes sin tocar el disco.


In [12]:
def resolve_raw_input(path: Path):
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            valid = [
                n for n in zf.namelist()
                if not n.startswith("__MACOSX") and (n.endswith(".csv") or n.endswith(".xlsx"))
            ]
            if not valid:
                raise ValueError(f"No se encontró archivo CSV/XLSX dentro de {path.name}")
            inner = valid[0]
            data = zf.read(inner)
            inner_ext = Path(inner).suffix.lower()
            return io.BytesIO(data), inner_ext, Path(inner).stem
    return path, path.suffix.lower(), path.stem


def decode_to_utf8_bytes(raw_bytes: bytes) -> bytes:
    if raw_bytes.startswith(b"\xef\xbb\xbf"):
        raw_bytes = raw_bytes[3:]
    try:
        raw_bytes.decode("utf-8")
        return raw_bytes
    except UnicodeDecodeError:
        return raw_bytes.decode("latin1").encode("utf-8")


## 3. Heurísticas de Reparación y Bitácora (Antes -> Después)

* `iter_clean_lines`: para `corrupted_csv`. Registra en bitácora el bloque de líneas crudas antes de corregir (`lineas_crudas_antes`) y la línea resultante limpia (`linea_corregida_despues`).
* `repair_xlsx`: para `corrupted_xlsx`. Desagrega las celdas de `NOMBRE_SERVICIO` con saltos de línea y registra la fila original y los registros generados.
* `iter_without_marker`: para `extra_line`. Aísla la fila atípica hacia `bitacora/{archivo}_fila_descartada.csv`.


In [13]:
from math import remainder
def iter_clean_lines(source_or_path, bitacora_path: Path | None = None):
    state = "normal"
    missing_first_column = None
    fecha_corte = None
    line_number = 0

    bitacora_file = None
    writer = None
    if bitacora_path:
        bitacora_file = bitacora_path.open("w", encoding="utf-8", newline="")
        writer = csv.writer(bitacora_file, delimiter=";")
        writer.writerow(["linea_inicio", "linea_fin", "tipo_anomalia", "lineas_crudas_antes", "linea_corregida_despues"])

    source = source_or_path.open("rb") if hasattr(source_or_path, "open") and not hasattr(source_or_path, "read") else source_or_path
    should_close = (source != source_or_path)

    raw_buffer = []
    start_line_num = 0

    try:
        for line in source:
            line_number += 1
            line_orig = line

            line = line.replace(b';";', b'";')
            line = line.replace(b'"""', b'"')
            quote_pos = line.find(b'"')

            if fecha_corte is None and b";" in line:
                token = line.split(b";", 1)[0].strip()
                if len(token) == 8 and token.isdigit():
                    fecha_corte = token

            has_single_quote = (quote_pos != -1 and line.find(b'"', quote_pos + 1) == -1)

            if state == "normal":
                if has_single_quote:
                    state = "corrupted"
                    start_line_num = line_number
                    raw_buffer = [line_orig.decode("latin1", errors="replace").rstrip("\r\n")]
                    line_fixed = line.replace(b'"', b"").replace(b"\t", b";")
                    if writer:
                        writer.writerow([start_line_num, line_number, "inicio_comilla_desfasada", raw_buffer[0], line_fixed.decode("latin1", errors="replace").rstrip("\r\n")])
                    yield line_fixed
                    continue
                else:
                    yield line
                    continue

            # Estado corrupto
            raw_buffer.append(line_orig.decode("latin1", errors="replace").rstrip("\r\n"))

            if not has_single_quote:
                if missing_first_column is None and fecha_corte:
                    missing_first_column = not line.startswith(fecha_corte)
                line = line.replace(b"\t", b";")
                if missing_first_column:
                    line = fecha_corte + b';' + line
                yield line
                continue

            # Cierre de bloque
            if missing_first_column:
                prefix_to_add = fecha_corte + b";"
                line = prefix_to_add + line
                quote_pos += len(prefix_to_add)
            prefix = line[:quote_pos]

            if b";;;;" in line[quote_pos + 1:]:
                num_tabs = prefix.count(b"\t")
                prefix = prefix.replace(b"\t", b";")

                next_line = next(source, b"")
                line_number += 1
                raw_buffer.append(next_line.decode("latin1", errors="replace").rstrip("\r\n"))

                if next_line.strip().startswith(b";;;;;;;"):
                    third_line = next(source, b"")
                    line_number += 1
                    raw_buffer.append(third_line.decode("latin1", errors="replace").rstrip("\r\n"))
                    if not third_line.strip().endswith(b";;;"):
                        state = "normal"
                        continue
                    first_semi = third_line.find(b";")
                    remainder = (
                        third_line[first_semi + 1:]
                        if first_semi != -1
                        else third_line
                    )
                    match = re.search(rb";{3,}|19000100", remainder)
                    if match:
                        remainder = remainder[:match.start()]

                    newline = b"\r\n" if third_line.endswith(b"\r\n") else b"\n"
                    assembled = prefix + remainder.rstrip(b"\r\n") + newline
                    if writer:
                        writer.writerow([start_line_num, line_number, "decimal_partido_3_lineas", " [SALTO] ".join(raw_buffer), assembled.decode("latin1", errors="replace").rstrip("\r\n")])
                    yield assembled
                else:
                    newline = b"\r\n" if next_line.endswith(b"\r\n") else b"\n"
                    next_line_fixed = next_line[:-len(newline)][:-num_tabs] + newline
                    if next_line_fixed.startswith(b"120"):
                        next_line_fixed = b"0" + next_line_fixed
                    assembled = prefix + next_line_fixed
                    if writer:
                        writer.writerow([start_line_num, line_number, "linea_partida_2_lineas", " [SALTO] ".join(raw_buffer), assembled.decode("latin1", errors="replace").rstrip("\r\n")])
                    yield assembled
            else:
                prefix = prefix.replace(b"\t", b";")
                assembled = prefix + line[quote_pos + 1:]
                if writer:
                    writer.writerow([start_line_num, line_number, "bloque_desfasado_tabs", " [SALTO] ".join(raw_buffer), assembled.decode("latin1", errors="replace").rstrip("\r\n")])
                yield assembled

            state = "normal"
            raw_buffer = []
    finally:
        if bitacora_file:
            bitacora_file.close()
        if should_close:
            try: source.close()
            except: pass


def iter_without_marker(source_or_path, extra_path: Path, marker: bytes):
    source = source_or_path.open("rb") if hasattr(source_or_path, "open") and not hasattr(source_or_path, "read") else source_or_path
    should_close_src = (source != source_or_path)
    extra = extra_path.open("wb") if hasattr(extra_path, "open") else extra_path
    try:
        for line in source:
            line = line.replace(b';";', b'";')
            if marker in line:
                extra.write(line)
            else:
                yield line
    finally:
        if should_close_src:
            try: source.close()
            except: pass
        if hasattr(extra_path, "open") and extra:
            try: extra.close()
            except: pass


def repair_xlsx(df: pl.DataFrame, bitacora_path: Path | None = None) -> pl.DataFrame:
    df.columns = COLUMNS
    nombre_idx = COLUMNS.index("NOMBRE_SERVICIO")
    next_column = COLUMNS[nombre_idx + 1]

    indices = (
        df.with_row_index("idx")
        .filter(pl.col("NOMBRE_SERVICIO").str.contains(r"\r|\n"))
        .get_column("idx")
        .to_list()
    )

    if bitacora_path and indices:
        all_corrupted_indices = set(indices)
        for idx in indices:
            if df[next_column][idx] is None and (idx + 1) < len(df):
                all_corrupted_indices.add(idx + 1)

        df_corrupted = df.with_row_index("fila_original").filter(
            pl.col("fila_original").is_in(sorted(all_corrupted_indices))
        )
        df_corrupted.write_csv(bitacora_path, separator=";")

    rows = []
    last_index = 0

    for start in indices:
        rows.extend(df.slice(last_index, start - last_index).rows())

        prefix = [df["FECHA_CORTE"][start], df["NRO_SERVICIO"][start]]
        records = [
            value.split("\t")
            for value in str(df["NOMBRE_SERVICIO"][start]).splitlines()
            if value.strip()
        ]

        needs_continuation = df[next_column][start] is None

        if needs_continuation:
            end = start + 1
            continuation = [df[column][end] for column in COLUMNS]
            if continuation[0] in {"240502", "240521"}:
                continuation[0] = "0" + continuation[0]

            last_record = records[-1]
            last_record[-1] += (str(continuation[0]) if continuation[0] is not None else "")
            last_record.extend(str(value) for value in continuation[1:] if value is not None)
            last_index = end + 1
        else:
            last_index = start + 1
            suffix = [df[column][start] for column in COLUMNS[nombre_idx + 1:]]
            last_record = records[-1]
            last_record.extend(str(value) if value is not None else None for value in suffix)

        for index, record in enumerate(records):
            row = prefix + record if index == 0 else record
            if len(row) != len(COLUMNS):
                raise ValueError(f"Fila {start} reconstruida con {len(row)} columnas")
            rows.append(row)

    rows.extend(df.slice(last_index).rows())
    return pl.DataFrame(rows, schema=COLUMNS, orient="row")


## 4. Lectores y Transformación

`read_input` despacha según el reader configurado en el catálogo (`corrupted_csv`, `corrupted_xlsx`, `extra_line`, etc.).
`transform` realiza la conversión de fechas, números y filtra las 14 columnas no analíticas.


In [14]:
def read_csv(source_or_path, reader, output_dir, stem="data", marker=None, separator=";", bitacora_dir=None):
    if bitacora_dir is None:
        bitacora_dir = output_dir / "bitacora"
    bitacora_dir.mkdir(parents=True, exist_ok=True)

    if reader == "corrupted_csv":
        bitacora_path = bitacora_dir / f"{stem}_corrupted_lines.csv"
        cleaned = b"".join(iter_clean_lines(source_or_path, bitacora_path=bitacora_path))
        source = io.BytesIO(decode_to_utf8_bytes(cleaned))
        has_header = True
    elif reader == "extra_line":
        extra_path = bitacora_dir / f"{stem}_fila_descartada.csv"
        cleaned = b"".join(iter_without_marker(source_or_path, extra_path, marker.encode()))
        source = io.BytesIO(decode_to_utf8_bytes(cleaned))
        has_header = False
    else:
        if hasattr(source_or_path, "read"):
            raw_bytes = source_or_path.read()
        else:
            raw_bytes = source_or_path.read_bytes()
        if separator == ";" and b";" not in raw_bytes[:4096] and b"," in raw_bytes[:4096]:
            separator = ","
        source = io.BytesIO(decode_to_utf8_bytes(raw_bytes))
        has_header = False

    return pl.read_csv(
        source, separator=separator, encoding="utf8", has_header=has_header,
        new_columns=COLUMNS, infer_schema=False, truncate_ragged_lines=True,
    )


def read_input(path: Path, spec: dict, output_dir: Path) -> pl.DataFrame:
    source, ext, stem = resolve_raw_input(path)
    reader = spec.get("reader", "csv")
    params = spec.get("parameters", {})
    marker = params.get("marker") or spec.get("marker")
    separator = params.get("separator") or spec.get("separator", ";")

    bitacora_dir = output_dir / "bitacora"
    bitacora_dir.mkdir(parents=True, exist_ok=True)

    if ext == ".csv" and reader == "xlsx":
        reader = "csv"
    elif ext == ".xlsx" and reader == "csv":
        reader = "xlsx"

    if reader in {"csv", "corrupted_csv", "extra_line"}:
        df = read_csv(source, reader, output_dir, stem=stem, marker=marker, separator=separator, bitacora_dir=bitacora_dir)
    else:
        columns_schema = spec.get("columns_schema") or params.get("columns_schema") or "legacy"
        schema_overrides = SCHEMA_LEGACY_STRING if columns_schema == "legacy" else SCHEMA_STANDARD_STRING
        df = pl.read_excel(source, schema_overrides=schema_overrides)
        if reader == "corrupted_xlsx":
            bitacora_path = bitacora_dir / f"{stem}_corrupted_rows.csv"
            df = repair_xlsx(df, bitacora_path=bitacora_path)
        else:
            rename_map = {}
            for c in df.columns:
                c_clean = c.strip().lower()
                if c_clean in NORMALIZED_MAP and NORMALIZED_MAP[c_clean] != c:
                    rename_map[c] = NORMALIZED_MAP[c_clean]
            if rename_map:
                df = df.rename(rename_map)

            if set(COLUMNS).issubset(set(df.columns)):
                df = df.select(COLUMNS)
            elif len(df.columns) == len(COLUMNS):
                df.columns = COLUMNS

    # Descartar cabecera repetida si vino incluida en los datos
    if len(df) > 0:
        first_tipo = str(df["ID_TIPO_DOC_FAC"][0]).strip().upper()
        first_periodo = str(df["PERIODO"][0]).strip()
        if first_tipo in {"ID_TIPO_DOC_FAC", "IDTIPODOCUMENTO"} or not first_periodo.isdigit():
            df = df.slice(1)

    return df


def transform(df: pl.DataFrame) -> pl.DataFrame:
    cartera_expr = pl.col("CARTERA").str.strip_chars().str.to_uppercase().str.slice(0, 1)
    expressions = [
        pl.col(column).str.replace_all("-", "").str.to_date("%Y%m%d", strict=False)
        if dtype == pl.Date else pl.col(column).cast(dtype)
        for column, dtype in SCHEMA_VAL.items()
    ]
    return (
        df
        .with_columns(cartera_expr)
        .with_columns(expressions)
        .with_columns(
            pl.when(pl.col("PERIODO") >= 10_000_000)
            .then(pl.col("PERIODO") // 100)
            .otherwise(pl.col("PERIODO"))
            .alias("PERIODO")
        )
        .drop(DROP_COLUMNS)
    )


## 5. Ejecución del Procesamiento por Distribuidora

Procesa automáticamente cada carpeta encontrada en  (, , etc.) según los metadatos de .
Detecta si el archivo Parquet de destino ya existe para omitirlo automáticamente (se puede forzar con ).


In [15]:
catalog = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
datasets = catalog.get("datasets", {})

# Parámetro para forzar reprocesamiento si se desea
FORCE_REPROCESS = False

# Mapeo de carpeta raw -> dataset_key
mapa_datasets = {}
for k, v in datasets.items():
    raw_storage = v.get("storage", {}).get("raw", "")
    folder_name = Path(raw_storage).name
    mapa_datasets[folder_name] = (k, v)

resultados_globales = []
tiempo_inicio_total = time.time()

for folder_name in carpetas_raw:
    if folder_name not in mapa_datasets:
        print(f"Saltando carpeta sin registro en metadatos: {folder_name}")
        continue
        
    dataset_key, spec = mapa_datasets[folder_name]
    raw_dir = RAW_BASE_DIR / folder_name
    interim_dir = INTERIM_BASE_DIR / folder_name
    bitacora_dir = interim_dir / "bitacora"
    interim_dir.mkdir(parents=True, exist_ok=True)
    bitacora_dir.mkdir(parents=True, exist_ok=True)
    
    sources = spec.get("sources", {})
    print(f"{'='*65}")
    print(f"Procesando distribuidora: {spec['name']} ({folder_name})")
    print(f"Períodos registrados:     {len(sources)}")
    print(f"Destino Parquet:          {interim_dir}")
    print(f"{'='*65}")
    
    for period, item in sources.items():
        filename = item.get("source", {}).get("filename")
        ingestion = item.get("ingestion", {"reader": "csv"})
        reader_type = ingestion.get("reader", "csv")
        
        # Buscar archivo en raw (soporta .zip, .csv, .xlsx)
        input_path = raw_dir / filename
        if not input_path.exists():
            stem = Path(filename).stem
            for ext in [".zip", ".csv", ".xlsx"]:
                cand = raw_dir / f"{stem}{ext}"
                if cand.exists():
                    input_path = cand
                    break
                    
        if not input_path.exists():
            print(f"  [{period}] Pendiente: {filename} no encontrado en raw.")
            continue
            
        output_path = interim_dir / f"{period}.parquet"
        
        # Omitir si ya existe y tiene contenido válido
        if output_path.exists() and output_path.stat().st_size > 0 and not FORCE_REPROCESS:
            peso_mb = output_path.stat().st_size / (1024 * 1024)
            filas = pl.scan_parquet(output_path).select(pl.len()).collect().item()
            resultados_globales.append({
                "distribuidora": folder_name,
                "periodo": period,
                "archivo_raw": input_path.name,
                "reader": reader_type,
                "filas": filas,
                "parquet_mb": round(peso_mb, 2),
                "segundos": 0.0,
                "estado": "OMITIDO_YA_EXISTE",
            })
            print(f"  [{period}] {input_path.name:<45} | {filas:>8,} filas | {peso_mb:>4.2f}MB | [Omitido, ya existe]")
            continue

        t0 = time.time()
        df_raw = read_input(input_path, ingestion, interim_dir)
        df_clean = transform(df_raw)
        df_clean.write_parquet(output_path)
        duracion = time.time() - t0
        peso_mb = output_path.stat().st_size / (1024 * 1024)
        
        resultados_globales.append({
            "distribuidora": folder_name,
            "periodo": period,
            "archivo_raw": input_path.name,
            "reader": reader_type,
            "filas": len(df_clean),
            "parquet_mb": round(peso_mb, 2),
            "segundos": round(duracion, 2),
            "estado": "PROCESADO",
        })
        print(f"  [{period}] {input_path.name:<45} | {len(df_clean):>8,} filas | {peso_mb:>4.2f}MB | {duracion:>4.2f}s")

tiempo_total = time.time() - tiempo_inicio_total
print(f"Procesamiento terminado en {tiempo_total:.2f}s ({tiempo_total / 60:.2f} min). Total períodos registrados: {len(resultados_globales)}")


Procesando distribuidora: Consumo Energético Clientes ENOSA (enosa)
Períodos registrados:     32
Destino Parquet:          /home/Yair/CodeProjects/MachineLearning/Project/data/interim/enosa
  [202401] DatosCompartidos202401.zip                    |  562,824 filas | 2.09MB | 0.74s
  [202402] DatosCompartido202402.xlsx                    |  562,468 filas | 2.29MB | 4.32s
  [202403] DatosCompartidos202403.zip                    |  564,928 filas | 2.32MB | 6.48s
  [202404] DatosCompartido202404.zip                     |  564,991 filas | 2.08MB | 6.53s
  [202405] DatosCompartido202405.zip                     |  566,779 filas | 2.06MB | 6.01s
  [202406] DatosCompartido202406.xlsx                    |  567,178 filas | 2.06MB | 4.10s
  [202407] DatosCompartido202407.csv                     |  568,829 filas | 2.05MB | 0.47s
  [202408] Consumo Energético clientes enosa Agosto 2024.csv |  570,057 filas | 2.07MB | 0.49s
  [202409] Consumo_Energetico_clientes_enosa_Setiembre_2024.csv |  570,874 fil

## 6. Auditoría y Control de Calidad

Revisión de volumen de datos procesados y chequeo de valores nulos en columnas críticas (`PERIODO`, `UBIGEO`, `IMPORTE`, `CONSUMO`).


In [16]:
df_resumen = pd.DataFrame(resultados_globales)

print("--- Resumen por distribuidora ---")
resumen_dist = df_resumen.groupby("distribuidora").agg(
    total_periodos=("periodo", "count"),
    total_filas=("filas", "sum"),
    peso_total_mb=("parquet_mb", "sum"),
    tiempo_total_s=("segundos", "sum")
).reset_index()
display(resumen_dist)

COLUMNAS_IGNORAR_NULOS = {
    "NOMBRE_SERVICIO",
    "FECHA_CONSUMO_HASTA",
    "FECHA_CONSUMO_DESDE",
    "FECHA_CORTE",
    "FECHA_EMISION",
    "FECHA_VENCIMIENTO"
}

for dist in df_resumen["distribuidora"].unique():

    d_dir = INTERIM_BASE_DIR / dist
    parquets = sorted(d_dir.glob("*.parquet"))

    print(f"\n{'=' * 100}")
    print(f"DISTRIBUIDORA: {dist}")
    print(f"{'=' * 100}")

    total_nulos = 0
    filas_dist = 0

    for p in parquets:

        df_p = pl.read_parquet(p)
        filas_dist += len(df_p)

        # ====================================================
        # Columnas que SÍ queremos auditar
        # ====================================================

        columnas_auditar = [
            col
            for col in df_p.columns
            if col not in COLUMNAS_IGNORAR_NULOS
        ]

        # ====================================================
        # Detectar NULL en cualquier columna auditada
        # ====================================================

        null_counts = (
            df_p
            .select(columnas_auditar)
            .null_count()
        )

        columnas_con_nulos = {
            col: count
            for col, count in zip(
                null_counts.columns,
                null_counts.row(0)
            )
            if count > 0
        }

        # Si no hay ningún NULL en las columnas relevantes,
        # pasar directamente al siguiente archivo.
        if not columnas_con_nulos:
            continue

        # ====================================================
        # Filas completas con al menos un NULL
        # ====================================================

        mask_nulos = pl.any_horizontal(
            pl.col(col).is_null()
            for col in columnas_auditar
        )

        filas_con_nulos = df_p.filter(mask_nulos)

        total_nulos += sum(columnas_con_nulos.values())

        print(f"\n⚠️ ARCHIVO: {p.name}")
        print(f"Filas totales: {len(df_p):,}")
        print(
            f"Filas con al menos un NULL: "
            f"{filas_con_nulos.height:,}"
        )

        print("\nColumnas con NULL:")

        for col, count in columnas_con_nulos.items():
            print(f"  - {col}: {count:,}")

        print("\nFilas completas con NULL:")

        # Mostrar máximo 50 para no saturar el notebook
        display(
            filas_con_nulos.head(50)
        )

    print(f"\nTotal filas de {dist}: {filas_dist:,}")

--- Resumen por distribuidora ---


,distribuidora,total_periodos,total_filas,peso_total_mb,tiempo_total_s
0,enosa,32,18518031,70.63,41.26
1,hidrandina,33,32923149,129.97,34.53



DISTRIBUIDORA: enosa

Total filas de enosa: 18,518,031

DISTRIBUIDORA: hidrandina

Total filas de hidrandina: 32,923,149


## 7. Registros de Bitácora (Antes -> Después)

Muestra de las evidencias forenses capturadas durante la corrección de anomalías.


In [17]:
for dist in carpetas_raw:
    b_dir = INTERIM_BASE_DIR / dist / "bitacora"
    if not b_dir.exists():
        continue
        
    archivos_b = sorted(b_dir.glob("*.csv"))
    print(f"\nBitácora de {dist} ({len(archivos_b)} archivos):")
    for f in archivos_b:
        print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
        
    # Muestra de corrupted_lines si existe (Antes -> Después)
    corrupted_csvs = list(b_dir.glob("*_corrupted_lines.csv"))
    if corrupted_csvs:
        ejemplo = corrupted_csvs[0]
        df_ej = pl.read_csv(ejemplo, separator=";", n_rows=3, ignore_errors=True)
        print(f"\nEjemplo Antes -> Después ({ejemplo.name}):")
        display(df_ej.to_pandas())



Bitácora de enosa (5 archivos):
 - Consumo_Energetico_clientes_enosa_Diciembre_2024_fila_descartada.csv (0.2 KB)
 - DatosCompartido202404_corrupted_rows.csv (318.5 KB)
 - DatosCompartido202405_corrupted_rows.csv (304.0 KB)
 - DatosCompartidos202401_corrupted_lines.csv (295.6 KB)
 - DatosCompartidos202403_corrupted_rows.csv (292.9 KB)

Ejemplo Antes -> Después (DatosCompartidos202401_corrupted_lines.csv):


,linea_inicio,linea_fin,tipo_anomalia,lineas_crudas_antes,linea_corregida_despues
0,85682,85682,inicio_comilla_desfasada,"20240131;0;""XXXXXXXXXXXXXa\tMXXXXXXXXXXXXX \t1...",20240131;0;XXXXXXXXXXXXXa;MXXXXXXXXXXXXX ;11;X...
1,85682,85731,bloque_desfasado_tabs,"20240131;0;""XXXXXXXXXXXXXa\tMXXXXXXXXXXXXX \t1...",20240131;0;IXXXXXXXXXXXXX;JXXXXXXXXXXXXXe;3;XX...
2,105193,105193,inicio_comilla_desfasada,"20240131;0;""XXXXXXXXXXXXXs\tMXXXXXXXXXXXXXL\t1...",20240131;0;XXXXXXXXXXXXXs;MXXXXXXXXXXXXXL;11;X...



Bitácora de hidrandina (1 archivos):
 - DatosAbiertos_consumohdna_202305_corrupted_lines.csv (102.9 KB)

Ejemplo Antes -> Después (DatosAbiertos_consumohdna_202305_corrupted_lines.csv):


,linea_inicio,linea_fin,tipo_anomalia,lineas_crudas_antes,linea_corregida_despues
0,388754,388754,inicio_comilla_desfasada,"20230531;0;""XXXXXXXXXXXXX.\tJXXXXXXXXXXXXX \t3...",20230531;0;XXXXXXXXXXXXX.;JXXXXXXXXXXXXX ;3;XX...
1,388754,388890,decimal_partido_3_lineas,"20230531;0;""XXXXXXXXXXXXX.\tJXXXXXXXXXXXXX \t3...",20230531;0;MXXXXXXXXXXXXXa;CXXXXXXXXXXXXXo;1;X...
2,678890,678890,inicio_comilla_desfasada,"20230531;0;""XXXXXXXXXXXXX.\tCXXXXXXXXXXXXXr\t1...",20230531;0;XXXXXXXXXXXXX.;CXXXXXXXXXXXXXr;11;X...
